<a href="https://colab.research.google.com/github/tseringTen/funnel-conversion-analysis/blob/main/funnel_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Quick recap of everything thats verified on this table:

Shape/structure confirmed
mql_id unique — no duplicate primary keys
Nulls handled: 60 blank origin values filled as "unknown", merged sensibly with 1,099 pre-existing "unknown" entries (documented distinction in your notes)
origin categories are clean — no casing/formatting inconsistencies
landing_page_id — 495 unique pages, extreme long-tail distribution (median 2 leads, max 912) — flagged for later analysis
first_contact_date — parsed to datetime, range validated

In [ ]:
import pandas as pd
raw_df=pd.read_csv('/content/olist_marketing_qualified_leads_dataset.csv')

In [ ]:
df=raw_df.copy()

In [ ]:
df.head(5)

,mql_id,first_contact_date,landing_page_id,origin
0,dac32acd4db4c29c230538b72f8dd87d,2018-02-01,88740e65d5d6b056e0cda098e1ea6313,social
1,8c18d1de7f67e60dbd64e3c07d7e9d5d,2017-10-20,007f9098284a86ee80ddeb25d53e0af8,paid_search
2,b4bc852d233dfefc5131f593b538befa,2018-03-22,a7982125ff7aa3b2054c6e44f9d28522,organic_search
3,6be030b81c75970747525b843c1ef4f8,2018-01-22,d45d558f0daeecf3cccdffe3c59684aa,email
4,5420aad7fec3549a85876ba1c529bd84,2018-02-21,b48ec5f3b04e9068441002a19df93c6c,organic_search


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8000 entries, 0 to 7999
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   mql_id              8000 non-null   object
 1   first_contact_date  8000 non-null   object
 2   landing_page_id     8000 non-null   object
 3   origin              7940 non-null   object
dtypes: object(4)
memory usage: 250.1+ KB


In [ ]:
df.dtypes # first_contact_date column is in object dtype and not datetime

,0
mql_id,object
first_contact_date,object
landing_page_id,object
origin,object


In [ ]:
df['mql_id'].duplicated().sum()   #confirmed mql_id uniqueness before joining, since the dataset docs flag that leads can originate from multiple landing pages.

np.int64(0)

In [ ]:
df.isnull().sum()   #origin column has 60 null values detected

,0
mql_id,0
first_contact_date,0
landing_page_id,0
origin,60


In [ ]:
df['origin']=df['origin'].fillna('unknown') #0.75% of leads had no recorded origin — likely untagged traffic — and were bucketed as 'unknown' rather than dropped or imputed,
   #                                         to avoid inflating or fabricating channel-level conversion rates

In [ ]:
df['origin'].value_counts(dropna=False) #"unknown" now consists two different things — Olist's own untracked traffic (1,099) and rows that were simply blank in the raw CSV (60)
#                                         Since,we don't know the channel for both, so merging them is fine.

,count
origin,
organic_search,2296
paid_search,1586
social,1350
unknown,1159
direct_traffic,499
email,493
referral,284
other,150
display,118


In [ ]:
df['landing_page_id'].nunique()

495

In [ ]:
df['landing_page_id'].value_counts().describe()  #max is 912 so 1 landing page alone drove over 11% of all leads —
#                                                worth understanding what made it so effective and whether it's still active.

,count
count,495.000000
mean,16.161616
std,71.622253
min,1.000000
25%,1.000000
50%,2.000000
75%,9.000000
max,912.000000


In [ ]:
df['landing_page_id'].value_counts().head(15)

,count
landing_page_id,
b76ef37428e6799c421989521c0e5077,912
22c29808c4f815213303f8933030604c,883
58326e62183c14b0c03085c33b9fdc44,495
88740e65d5d6b056e0cda098e1ea6313,445
ce1a65abd0973638f1c887a6efcfa82d,394
40dec9f3d5259a3d2dbcdab2114fae47,330
f017be4dbf86243af5c1ebed0cff36a2,310
e492ee5eaf1697716985cc6f33f9cd9b,291
a7982125ff7aa3b2054c6e44f9d28522,156


In [ ]:
df['first_contact_date']=pd.to_datetime(df['first_contact_date'])   #date parsing
df['first_contact_date'].min(),df['first_contact_date'].max()

(Timestamp('2017-06-14 00:00:00'), Timestamp('2018-05-31 00:00:00'))

In [ ]:
df.to_csv('MQL_DATASET.csv',index=False)
from google.colab import files
files.download('MQL_DATASET.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>